In [1]:
%reload_ext autoreload
%autoreload 2

import warnings, os, sys
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from rdkit import Chem
from rdkit.Chem import PandasTools
# PandasTools.RenderImagesInAllDataFrames(images=True)
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 700, 500

sys.path.append(os.path.abspath(".."))
from new_geass import Geass

In [2]:
df_NPA = pd.read_csv('../data/NPAtlas_download_2024_09.tsv', sep='\t')
PandasTools.AddMoleculeColumnToFrame(df_NPA, smilesCol='compound_smiles')

In [3]:
# Fragment molecules and write to file in batches
with open('NPA_fragments.csv', 'a') as f:
    geass = Geass()

    buffer = []
    for row in tqdm(df_NPA.iloc[10000:20000].itertuples(), total=10000):
        try:
            frags = geass.get_fragments(row.ROMol)
            frag_str = ','.join(Chem.MolToSmiles(f) for f in frags)
            mu_str = ','.join(f.GetProp('MU_ID') for f in frags)
        except Exception as e:
            frag_str, mu_str = "ERROR", ""

        full_str = f'{row.npaid},{row.compound_smiles},"[{frag_str}]","[{mu_str}]"\n'
        buffer.append(full_str)

        if len(buffer) >= 500:
            f.writelines(buffer)
            buffer.clear()

    if buffer:
        f.writelines(buffer)

  0%|          | 0/10000 [00:00<?, ?it/s]

Annotation resources loaded.


In [ ]:
# Assign KEGG- or U-ID to fragments
df_MUDB = pd.read_pickle('../data/MUDB.pkl')
df_frags = pd.read_csv('geass_NPA_fragments.csv', header=None)

df_frags.columns = ['ID', 'SMILES', 'Fragments']
df_frags['Fragments'] = df_frags['Fragments'].str.strip('[]').str.split(',')

df_unfragmented = df_frags[df_frags['Fragments'].apply(len) == 1]
df_fragmented = df_frags[df_frags['Fragments'].apply(len) > 1]
# df_error = df_unfrag[df_unfrag['Fragments'].apply(lambda x: 'ERROR' in x)]

uu_list = {}
id_list = []
for frag_list in tqdm(df_fragmented['Fragments'].tolist()):
    ids = []
    for smi in frag_list:
        inchi = Chem.MolToInchi(Chem.MolFromSmiles(smi))
        mu = df_MUDB[df_MUDB['InChI'] == inchi]

        if mu.shape[0] > 0:
            id = mu['ID'].values[0]
        else:
            # Mark as unknown compound
            id = uu_list.setdefault(smi, f"U{len(uu_list)+1:05d}")
        ids.append(id)

    id_list.append(ids)
df_fragmented['Frag_IDs'] = id_list

df_fragmented.to_pickle('NPA_fragmented.pkl')
df_unfragmented.to_pickle('NPA_unfragmented.pkl')

# Create the unknown unit library
df_UUL = pd.DataFrame([(v, k) for k, v in uu_list.items()], columns=['UID', 'ISOSMILES'])
PandasTools.AddMoleculeColumnToFrame(df_UUL, smilesCol='ISOSMILES')
df_UUL = df_UUL.dropna(subset=['ROMol'])
df_UUL.drop(columns=['ROMol']).to_csv('../data/UUDB.csv', index=False)
df_UUL.to_pickle('../data/UUDB.pkl')